# DE-DNN-IDS on Google Colab

Differential Evolution optimised DNN for intrusion detection on
CSE-CIC-IDS2018.

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`. On CPU the
search takes most of a day; the whole reason to use Colab is the GPU.

Run the cells in order. Sections 1-5 are setup and take about 10 minutes.
Section 6 is the experiment.

**Nothing external to set up.** No Kaggle account, no API token, no dataset
upload. The notebook downloads the data itself from a public endpoint. The
Google Drive mount in section 5 is the only sign-in, and it is optional.

> **Colab disconnects.** Free sessions drop after roughly 90 minutes idle and
> cap out around 12 hours. With Drive mounted, the search writes its state
> there and resumes from it, so a disconnect costs you one generation, not the
> whole run. If you get dropped: reconnect, re-run cells 1-5, then re-run the
> same search cell. It picks up where it stopped.

## 1. Confirm the GPU is actually attached

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                      "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
      or "NO GPU - go to Runtime > Change runtime type > T4 GPU")

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print(f"TensorFlow {tf.__version__}, GPUs visible: {gpus}")
assert gpus, "No GPU visible to TensorFlow. Fix the runtime type before going on."

## 2. Get the code

In [ ]:
import os
REPO = "https://github.com/von-moyo/de-dnn-ids.git"
if not os.path.exists("/content/de-dnn-ids"):
    !git clone -q {REPO} /content/de-dnn-ids
else:
    !git -C /content/de-dnn-ids pull -q
%cd /content/de-dnn-ids
!git log --oneline -1

## 3. Dependencies

**Do not** run `pip install -r requirements.txt` here. That file pins
`numpy<2.1` for local reproducibility, and applying it in Colab downgrades the
NumPy that the pre-installed TensorFlow was built against — which forces a
runtime restart and can leave TF broken. Colab already ships everything except,
usually, `pyarrow`. Install only what is genuinely missing.

In [ ]:
import importlib

need = []
for mod, pkg in [("pyarrow", "pyarrow"), ("imblearn", "imbalanced-learn"),
                 ("seaborn", "seaborn"), ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)

if need:
    print("installing:", need)
    !pip install -q {" ".join(need)}
else:
    print("all dependencies already present")

import numpy, pandas, sklearn, pyarrow
print(f"numpy {numpy.__version__} | pandas {pandas.__version__} | "
      f"sklearn {sklearn.__version__} | pyarrow {pyarrow.__version__}")

## 4. Fetch the dataset

**No Kaggle account, no API token, no `kaggle.json`.** Kaggle serves public
datasets over an anonymous endpoint, so an ordinary download is all it takes:

```
https://www.kaggle.com/api/v1/datasets/download/dhoogla/csecicids2018
```

That is a 634 MB zip of the deduplicated parquet build of CSE-CIC-IDS2018 —
the same build every number in the write-up was measured on.

It goes to Colab's local disk rather than Drive: local reads are much faster
during training, and the file is always re-downloadable. The download resumes
if the connection drops, and the cell skips the whole step if the captures are
already there.

In [ ]:
%%time
import glob, os, subprocess, zipfile

URL = "https://www.kaggle.com/api/v1/datasets/download/dhoogla/csecicids2018"
ZIP = "data/csecicids2018.zip"

os.makedirs("data", exist_ok=True)


def zip_is_complete(path):
    """A truncated download has no readable central directory."""
    try:
        with zipfile.ZipFile(path) as zf:
            return len(zf.namelist()) > 0
    except Exception:
        return False


if len(glob.glob("data/*.parquet")) >= 10:
    print("captures already on disk, skipping download")
else:
    # -C - resumes a partial file, so a dropped connection costs only the
    # remainder rather than the whole 634 MB.
    for attempt in range(1, 4):
        if zip_is_complete(ZIP):
            break
        print(f"download attempt {attempt} of 3 ...")
        subprocess.call(["curl", "-L", "--retry", "5", "--retry-delay", "5",
                         "-C", "-", "-o", ZIP, URL])

    if not zip_is_complete(ZIP):
        raise RuntimeError(
            "Download did not finish. Just re-run this cell - it resumes from "
            "where it stopped rather than starting over.")

    print("\nextracting ...")
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall("data")
    os.remove(ZIP)

In [ ]:
# Kaggle occasionally serves large members individually zipped. Extract any
# nested archives to the name the loader expects, so it reads the extracted
# copy and ignores the archive rather than loading both.
import glob, os, zipfile, shutil

for z in sorted(glob.glob("data/*.zip")):
    target = z[:-4]
    if os.path.exists(target):
        continue
    with zipfile.ZipFile(z) as zf:
        inner = [n for n in zf.namelist()
                 if n.lower().endswith((".parquet", ".csv"))]
        if len(inner) == 1:
            with zf.open(inner[0]) as src, open(target, "wb") as dst:
                shutil.copyfileobj(src, dst, 1 << 20)
            print(f"extracted {os.path.basename(target)}")

caps = sorted(glob.glob("data/*.parquet"))
print(f"{len(caps)} capture files:")
for c in caps:
    print(f"  {os.path.basename(c):<66} {os.path.getsize(c)/1e6:7.1f} MB")
assert len(caps) >= 10, "expected 10 capture files - re-run the download cell"

## 5. Somewhere to put the results

Colab wipes local disk when the session ends, so results **and the DE
checkpoint** are safer on Drive: a disconnect then costs one generation instead
of the entire search.

Mounting Drive is the one step that asks you to sign in to Google. It is
optional — set `USE_DRIVE = False` below and everything lands on local disk
instead. The run works exactly the same; you only lose the ability to resume
after a disconnect. The dataset stays on local disk either way.

In [ ]:
# Set to False to skip the Google sign-in entirely. Results then go to Colab's
# local disk and are LOST when the session ends, so a disconnect means starting
# the search over instead of resuming from the checkpoint.
USE_DRIVE = True

import os

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/de-dnn-ids"
else:
    BASE = "/content/results"
    print("Drive skipped - results are on local disk and vanish on disconnect.")

STAGE1 = f"{BASE}/stage1"
FINAL = f"{BASE}/final"
CKPT = f"{BASE}/de_checkpoint.json"
for d in (STAGE1, FINAL):
    os.makedirs(d, exist_ok=True)
print(f"results    -> {BASE}\ncheckpoint -> {CKPT}")

## 6. Run

### 6a. Smoke test (~8 minutes)

Exercises the whole pipeline on a tiny sample. Confirms the data loaded, the
GPU is being used, and the class handling is right before you commit hours.
Expect: 3 classes dropped, `classes=12`, no `<-- TOO FEW` flags.

In [ ]:
%%time
!python de_dnn_ids.py --data_dir ./data --mode multiclass \
    --out_dir {BASE}/smoke \
    --max_per_class 2000 --min_class_rows 200 \
    --pop_size 4 --generations 1 --fitness_epochs 3 --final_epochs 5

### 6b. Noise floor - how large a difference can this experiment even see?

Run this **before** paying for a longer search.

`seed_variance.py` retrains the same configuration five times, changing only
the weight initialisation, and reports the spread. It then compares the DE
winner against the hand-tuned baseline and says whether the gap between them
clears that spread.

Why it matters for this dataset:

* the stage-1 DE winner scored macro-F1 **0.9041** against the baseline's
  **0.9019** - a gap of **0.0022**;
* **8 of the 12 classes already sit at F1 >= 0.99** and cannot move, so
  two-thirds of macro-F1 is frozen;
* **Brute Force -XSS has 46 test rows**, so one row landing differently moves
  macro-F1 by 0.0015 - most of the gap being argued over;
* the best of the *random* initial population (generation 0) already beat the
  hand-tuned baseline, and the search then plateaued from generation 4.

If run-to-run noise turns out to be the same size as the gap, no population
size and no number of generations rescues the comparison - the search would be
chasing differences the measurement cannot resolve. That is a real finding and
worth reporting, but it means changing the *experiment*, not the DE settings.

Ten trainings. The verdict is printed at the end.

In [ ]:
%%time
!python seed_variance.py --data_dir ./data --mode multiclass \
    --config report/stage1/best_config.json \
    --config baseline_config.json \
    --out_dir {BASE}/variance \
    --seeds 5 --epochs 100 \
    --max_per_class 20000 --min_class_rows 200

In [ ]:
import os
from IPython.display import Image, display

p = f"{BASE}/variance/seed_variance.png"
if os.path.exists(p):
    display(Image(p))
else:
    print("no figure at", p)

### 6c. Stage 1 - the DE search

Settings changed from the first run, for two reasons the artefacts pointed at:

**`--fitness_epochs 8` was the bigger problem.** `fitness()` sets early
stopping with patience 5, so an 8-epoch cap ends nearly every candidate before
the callback can act - the cap does the budgeting, not the convergence. The
winner is then retrained for 100 epochs. DE was therefore ranking candidates on
"best after 8 epochs", which is not the objective the final model is judged on,
and it quietly favours fast-starting configurations (high learning rate, small
batch) over ones that are better fully trained. 25 lets early stopping decide.

**`--pop_size 10` is below the guidance in the code's own docstring** (roughly
10x the dimensionality, so ~60 here, with 20 a reasonable compromise). At 10,
DE/rand/1 has few distinct donor triples to draw on and diversity collapses
early - consistent with the plateau from generation 4.

Cost: 20 + 20x12 = **260 evaluations against the previous 110**, each with a
larger epoch budget. Early stopping claws some of that back. Time the first
generation on your own GPU before trusting any estimate.

> **If the noise floor in 6b came back large**, add `--fitness_repeats 2`. It
> trains each candidate twice and averages, halving fitness noise at double the
> cost. Do not add it blindly - check 6b first.

> **The old checkpoint will not load.** A population of 10 cannot be grafted
> onto a run configured for 20; the script refuses rather than corrupting the
> search. Uncomment the `rm` line below to start fresh.

In [ ]:
%%time
# The population size changed, so the saved checkpoint no longer matches.
# Uncomment to discard it and start a fresh search:
# !rm -f {CKPT}

!python de_dnn_ids.py --data_dir ./data --mode multiclass \
    --out_dir {STAGE1} --checkpoint {CKPT} \
    --max_per_class 20000 --min_class_rows 200 \
    --pop_size 20 --generations 12 --fitness_epochs 25 --seed 42

### 6d. Stage 2 — train the winner

Skips the search and trains one network on 10x the data, using the winning
configuration.

`--load_config` replays only the *hyperparameters*. The data flags are **not**
stored in it, so `--max_per_class` and `--min_class_rows` must be repeated here
or you evaluate a different class set than the one DE tuned for.

In [ ]:
%%time
!python de_dnn_ids.py --data_dir ./data --mode multiclass \
    --out_dir {FINAL} \
    --max_per_class 200000 --min_class_rows 200 \
    --load_config {STAGE1}/best_config.json

## 7. Results

In [ ]:
import json, os
from IPython.display import Image, display

with open(f"{FINAL}/metrics.json") as fh:
    m = json.load(fh)

print("==== TEST METRICS ====")
for k, v in m.items():
    if k != "_sampling":
        print(f"{k:>26}: {v:.4f}")

print("\nHeadline for the writeup: macro F1 =", round(m["f1"], 4))
print("Operational false-alarm rate =",
      round(m.get("benign_false_alarm_rate", float('nan')), 4))
print("\nNOTE:", m.get("_sampling", {}).get("caveat", ""))

print("\n" + open(f"{FINAL}/classification_report.txt").read())
for fig in ("de_convergence.png", "confusion_matrix.png"):
    p = f"{STAGE1}/{fig}" if fig.startswith("de_") else f"{FINAL}/{fig}"
    if os.path.exists(p):
        display(Image(p))

---

## Reading the numbers

**Quote `benign_false_alarm_rate`, not `fpr`.** `fpr` is a macro one-vs-rest
average over all 12 classes; a dozen easy attack classes with almost no false
positives dilute it toward zero even when most benign traffic is being
misclassified. `benign_false_alarm_rate` is the fraction of genuinely benign
flows flagged as some attack, which is what the alert-fatigue argument is about.

**Accuracy and FPR are not operational estimates.** `--max_per_class` rebalances
the test set, so it does not carry real traffic's 80% benign prior. Macro
precision/recall/F1 weight classes equally and are unaffected. The caveat is
recorded in `metrics.json` under `_sampling`.

**Benign vs Infiltration is the hard case.** They are near-indistinguishable in
this dataset and the model confuses them in both directions. Expect that pair to
dominate the error budget; it is a property of CSE-CIC-IDS2018, not a bug.

**Dataset build.** This uses the deduplicated parquet redistribution, which has
6,659,532 rows against the raw release's ~16M. Deduplication collapses
FTP-BruteForce to 53 rows, DoS-SlowHTTPTest to 55 and SQL Injection to 85 —
`--min_class_rows 200` drops those three as unscoreable. Published results
almost all use the raw build, where duplicate flows appear in both train and
test and inflate the figures. **Your numbers will be lower and are not directly
comparable.** Say so explicitly. See `README.md` for the full comparison.

**A difference has to clear its own noise.** Section 6b exists because macro-F1
moves between identical runs. Quote the DE-versus-baseline gap only alongside
that spread; a gap smaller than the run-to-run standard deviation is not
evidence that the search found anything. Reporting the noise floor is the
honest version of this experiment, and most published DE-for-IDS comparisons
never do it.